# EA-TTM — Adapter Training and Validation

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 08 — EA-TTM adapter training and validation  
**Scope:** Train only the shock-aware residual adapter while the complete HP-TTM remains frozen, select the best checkpoint on validation, and quantify overall and extreme-regime performance.

The locked final-test interval remains untouched.

## Frozen base model

The HP-TTM base forecast is fixed:

$$
\hat y_{\mathrm{HP},h}.
$$

The trainable EA-TTM output is:

$$
\hat y_{\mathrm{EA},h}
=
\hat y_{\mathrm{HP},h}
+
g_h\Delta_h.
$$

Only the 2,466-parameter adapter is optimized.

## Multi-task training objective

EA-TTM has two coupled objectives:

1. improve the final retail forecast;
2. make the Extreme Gate learn the training-defined extreme regime.

The raw forecast loss is:

$$
\mathcal L_{\mathrm{forecast}}
=
\operatorname{MSE}
\left(
\hat{\mathbf Y}_{\mathrm{EA}},
\mathbf Y
\right).
$$

The gate loss is standard binary cross-entropy with logits:

$$
\mathcal L_{\mathrm{gate}}
=
\operatorname{BCEWithLogits}
\left(
\ell_{t,h},
e_{t,h}
\right),
$$

where $\ell_{t,h}$ is the gate logit and $e_{t,h}$ is the Phase 07 extreme label.

To avoid an arbitrary numerical dominance caused only by different loss scales, both losses are normalized by fixed training-reference quantities.

The forecasting reference is the frozen HP-TTM training MSE:

$$
M_{\mathrm{HP}}
=
\operatorname{MSE}
\left(
\hat{\mathbf Y}_{\mathrm{HP}},
\mathbf Y
\right).
$$

The gate reference is the entropy of the training extreme prior $\pi_e$:

$$
H(\pi_e)
=
-
\pi_e\log(\pi_e)
-
(1-\pi_e)\log(1-\pi_e).
$$

The optimized objective is:

$$
\boxed{
\mathcal L
=
\frac{
\mathcal L_{\mathrm{forecast}}
}{
M_{\mathrm{HP}}
}
+
\frac{
\mathcal L_{\mathrm{gate}}
}{
H(\pi_e)
}
}
$$

This gives forecasting and gate supervision comparable dimensionless scales without introducing an arbitrary task-weight coefficient.

## Validation checkpoint selection

EA-TTM is intended to improve forecasting generally **and** under extreme regimes.

Let:

$$
R_{\mathrm{all}}
=
\frac{
\mathrm{MSE}_{\mathrm{EA,all}}
}{
\mathrm{MSE}_{\mathrm{HP,all}}
}
$$

and:

$$
R_{\mathrm{ext}}
=
\frac{
\mathrm{MSE}_{\mathrm{EA,ext}}
}{
\mathrm{MSE}_{\mathrm{HP,ext}}
}.
$$

The primary validation selection score is:

$$
S_{\mathrm{balanced}}
=
\frac{
R_{\mathrm{all}}+R_{\mathrm{ext}}
}{
2
}.
$$

Lower is better.

This rule is fixed before training and gives equal relative importance to overall forecasting quality and extreme-regime forecasting quality. The best-overall and best-extreme checkpoints are also recorded for diagnostics, but the **balanced checkpoint is the primary EA-TTM model** used in later locked evaluation.

## Optimization contract

The adapter is small and its input features are already standardized or bounded. The optimizer is therefore:

- AdamW;
- initial learning rate $10^{-3}$;
- weight decay $10^{-4}$;
- batch size 512;
- gradient clipping at norm 1.0;
- maximum 30 epochs.

A `ReduceLROnPlateau` scheduler halves the learning rate after two validation plateaus. Early stopping occurs after six epochs without improvement in the balanced validation score.

The correction output begins at zero, so training starts exactly from HP-TTM.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch import nn

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

PHASE05_CACHE_DIR = (
    PROJECT_ROOT
    / "runs"
    / "05_decoder_crossfeed"
    / "cache"
)

PHASE06_FEATURE_DIR = (
    PROJECT_ROOT
    / "runs"
    / "06_retail_features"
    / "features"
)

PHASE07_TARGET_DIR = (
    PROJECT_ROOT
    / "runs"
    / "07_ea_ttm_architecture"
    / "targets"
)

PHASE07_INITIAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ea_ttm_adapter_initial"
)

RUN_DIR = (
    PROJECT_ROOT
    / "runs"
    / "08_ea_ttm_training"
)

CHECKPOINT_DIR = RUN_DIR / "checkpoints"

FINAL_ADAPTER_DIR = (
    PROJECT_ROOT
    / "models"
    / "ea_ttm_adapter"
)

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
    CHECKPOINT_DIR,
    FINAL_ADAPTER_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "phase04_summary": (
        METADATA_DIR
        / "04_RUN_SUMMARY.json"
    ),
    "phase05_summary": (
        METADATA_DIR
        / "05_RUN_SUMMARY.json"
    ),
    "phase05_cache_manifest": (
        METADATA_DIR
        / "05_CACHE_MANIFEST.json"
    ),
    "decoder_alignment_weights": (
        METADATA_DIR
        / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
    ),
    "phase06_summary": (
        METADATA_DIR
        / "06_RUN_SUMMARY.json"
    ),
    "phase07_summary": (
        METADATA_DIR
        / "07_RUN_SUMMARY.json"
    ),
    "phase07_extreme_contract": (
        METADATA_DIR
        / "07_EXTREME_TARGET_CONTRACT.json"
    ),
    "phase07_architecture_contract": (
        METADATA_DIR
        / "07_EA_TTM_ARCHITECTURE_CONTRACT.json"
    ),
    "adapter_initial_state": (
        PHASE07_INITIAL_MODEL_DIR
        / "adapter_initial_state.pt"
    ),
    "adapter_config": (
        PHASE07_INITIAL_MODEL_DIR
        / "adapter_config.json"
    ),
    "train_decoder_hidden": (
        PHASE05_CACHE_DIR
        / "train_decoder_hidden.npy"
    ),
    "train_base_forecast": (
        PHASE05_CACHE_DIR
        / "train_base_forecast.npy"
    ),
    "validation_decoder_hidden": (
        PHASE05_CACHE_DIR
        / "validation_decoder_hidden.npy"
    ),
    "validation_base_forecast": (
        PHASE05_CACHE_DIR
        / "validation_base_forecast.npy"
    ),
    "train_recent_features": (
        PHASE06_FEATURE_DIR
        / "train_recent_demand_features.npy"
    ),
    "validation_recent_features": (
        PHASE06_FEATURE_DIR
        / "validation_recent_demand_features.npy"
    ),
    "train_future_covariates": (
        PHASE06_FEATURE_DIR
        / "train_future_covariates.npy"
    ),
    "validation_future_covariates": (
        PHASE06_FEATURE_DIR
        / "validation_future_covariates.npy"
    ),
    "train_extreme_labels": (
        PHASE07_TARGET_DIR
        / "train_extreme_labels.npy"
    ),
    "validation_extreme_labels": (
        PHASE07_TARGET_DIR
        / "validation_extreme_labels.npy"
    ),
    "train_residual_targets": (
        PHASE07_TARGET_DIR
        / "train_residual_targets.npy"
    ),
    "validation_residual_targets": (
        PHASE07_TARGET_DIR
        / "validation_residual_targets.npy"
    ),
}

preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    item["path"]
    for item in preflight.values()
    if not item["exists"]
]

assert not missing_artifacts, (
    "Phase 08 prerequisites are incomplete: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


## Recover locked contracts

In [3]:
phase04_summary = json.loads(
    (
        METADATA_DIR
        / "04_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase05_summary = json.loads(
    (
        METADATA_DIR
        / "05_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

cache_manifest = json.loads(
    (
        METADATA_DIR
        / "05_CACHE_MANIFEST.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase06_summary = json.loads(
    (
        METADATA_DIR
        / "06_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

phase07_summary = json.loads(
    (
        METADATA_DIR
        / "07_RUN_SUMMARY.json"
    ).read_text(
        encoding="utf-8"
    )
)

extreme_contract = json.loads(
    (
        METADATA_DIR
        / "07_EXTREME_TARGET_CONTRACT.json"
    ).read_text(
        encoding="utf-8"
    )
)

architecture_contract = json.loads(
    (
        METADATA_DIR
        / "07_EA_TTM_ARCHITECTURE_CONTRACT.json"
    ).read_text(
        encoding="utf-8"
    )
)

adapter_config = json.loads(
    (
        PHASE07_INITIAL_MODEL_DIR
        / "adapter_config.json"
    ).read_text(
        encoding="utf-8"
    )
)

DATASET_SHA256 = phase07_summary["dataset_sha256"]
SPLIT_SHA256 = phase07_summary["split_sha256"]

FEATURE_CONTRACT_SHA256 = (
    phase07_summary[
        "feature_contract_sha256"
    ]
)

CROSSFEED_CONTRACT_SHA256 = (
    phase07_summary[
        "crossfeed_contract_sha256"
    ]
)

EXTREME_CONTRACT_SHA256 = (
    phase07_summary[
        "extreme_contract_sha256"
    ]
)

ARCHITECTURE_CONTRACT_SHA256 = (
    phase07_summary[
        "architecture_contract_sha256"
    ]
)

TRAINING_SAMPLE_COUNT = int(
    cache_manifest[
        "train_decoder_hidden"
    ]["shape"][0]
)

VALIDATION_SAMPLE_COUNT = int(
    cache_manifest[
        "validation_decoder_hidden"
    ]["shape"][0]
)

FORECAST_HORIZON = 28

assert DATASET_SHA256 == phase05_summary["dataset_sha256"]
assert DATASET_SHA256 == phase06_summary["dataset_sha256"]
assert SPLIT_SHA256 == phase05_summary["split_sha256"]
assert SPLIT_SHA256 == phase06_summary["split_sha256"]
assert TRAINING_SAMPLE_COUNT == 121_960
assert VALIDATION_SAMPLE_COUNT == 30_490
assert architecture_contract["input_dimension"] == 141
assert architecture_contract["encoder_width"] == 16
assert architecture_contract["correction_hidden_width"] == 8
assert architecture_contract["adapter_trainable_parameters"] == 2466
assert extreme_contract["locked_final_test_used"] is False

## Reproducibility state

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), (
    "CUDA GPU is required for Phase 08."
)

DEVICE = torch.device("cuda")

## Load cached inputs and targets

All large arrays are memory-mapped from Drive. The frozen HP-TTM is not executed during adapter training.

In [5]:
train_decoder_hidden = np.load(
    PHASE05_CACHE_DIR
    / "train_decoder_hidden.npy",
    mmap_mode="r",
)

train_base_forecast = np.load(
    PHASE05_CACHE_DIR
    / "train_base_forecast.npy",
    mmap_mode="r",
)

validation_decoder_hidden = np.load(
    PHASE05_CACHE_DIR
    / "validation_decoder_hidden.npy",
    mmap_mode="r",
)

validation_base_forecast = np.load(
    PHASE05_CACHE_DIR
    / "validation_base_forecast.npy",
    mmap_mode="r",
)

train_recent_features = np.load(
    PHASE06_FEATURE_DIR
    / "train_recent_demand_features.npy",
    mmap_mode="r",
)

validation_recent_features = np.load(
    PHASE06_FEATURE_DIR
    / "validation_recent_demand_features.npy",
    mmap_mode="r",
)

train_future_covariates = np.load(
    PHASE06_FEATURE_DIR
    / "train_future_covariates.npy",
    mmap_mode="r",
)

validation_future_covariates = np.load(
    PHASE06_FEATURE_DIR
    / "validation_future_covariates.npy",
    mmap_mode="r",
)

train_extreme_labels = np.load(
    PHASE07_TARGET_DIR
    / "train_extreme_labels.npy",
    mmap_mode="r",
)

validation_extreme_labels = np.load(
    PHASE07_TARGET_DIR
    / "validation_extreme_labels.npy",
    mmap_mode="r",
)

train_residual_targets = np.load(
    PHASE07_TARGET_DIR
    / "train_residual_targets.npy",
    mmap_mode="r",
)

validation_residual_targets = np.load(
    PHASE07_TARGET_DIR
    / "validation_residual_targets.npy",
    mmap_mode="r",
)

alignment = np.load(
    METADATA_DIR
    / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
)

selected_weight = alignment[
    "selected_weight"
].astype(
    np.float32,
    copy=False,
)

assert train_decoder_hidden.shape == (
    121_960,
    1,
    9,
    128,
)

assert validation_decoder_hidden.shape == (
    30_490,
    1,
    9,
    128,
)

assert train_recent_features.shape == (
    121_960,
    5,
)

assert validation_recent_features.shape == (
    30_490,
    5,
)

assert train_future_covariates.shape == (
    121_960,
    28,
    8,
)

assert validation_future_covariates.shape == (
    30_490,
    28,
    8,
)

assert train_extreme_labels.shape == (
    121_960,
    28,
)

assert validation_extreme_labels.shape == (
    30_490,
    28,
)

assert train_residual_targets.shape == (
    121_960,
    28,
)

assert validation_residual_targets.shape == (
    30_490,
    28,
)

assert selected_weight.shape == (
    28,
    9,
    128,
)

## Recover actual targets from frozen-base residual targets

Phase 07 defined:

$$
r^*_{t,h}
=
y_{t+h}
-
\hat y_{\mathrm{HP},t,h}.
$$

Therefore:

$$
y_{t+h}
=
\hat y_{\mathrm{HP},t,h}
+
r^*_{t,h}.
$$

In [6]:
training_targets = (
    np.asarray(
        train_base_forecast,
        dtype=np.float32,
    )
    + np.asarray(
        train_residual_targets,
        dtype=np.float32,
    )
).astype(
    np.float32
)

validation_targets = (
    np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    )
    + np.asarray(
        validation_residual_targets,
        dtype=np.float32,
    )
).astype(
    np.float32
)

assert np.isfinite(
    training_targets
).all()

assert np.isfinite(
    validation_targets
).all()

## EA-TTM adapter

In [7]:
class RetailShockAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int = 141,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
        extreme_prior: float = 0.05,
    ) -> None:
        super().__init__()

        self.input_dim = int(
            input_dim
        )

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.extreme_gate = nn.Linear(
            encoder_width,
            1,
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        self._initialize_output_heads(
            extreme_prior
        )

    def _initialize_output_heads(
        self,
        extreme_prior: float,
    ) -> None:
        nn.init.zeros_(
            self.extreme_gate.weight
        )

        prior_logit = math.log(
            extreme_prior
            / (1.0 - extreme_prior)
        )

        nn.init.constant_(
            self.extreme_gate.bias,
            prior_logit,
        )

        nn.init.zeros_(
            self.correction_head[-1].weight
        )

        nn.init.zeros_(
            self.correction_head[-1].bias
        )

    def forward(
        self,
        recent_features: torch.Tensor,
        decoder_features: torch.Tensor,
        future_covariates: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        horizon = decoder_features.shape[1]

        recent_expanded = (
            recent_features
            .unsqueeze(1)
            .expand(
                -1,
                horizon,
                -1,
            )
        )

        adapter_input = torch.cat(
            [
                recent_expanded,
                decoder_features,
                future_covariates,
            ],
            dim=-1,
        )

        shock_representation = (
            self.shock_encoder(
                adapter_input
            )
        )

        gate_logits = (
            self.extreme_gate(
                shock_representation
            ).squeeze(-1)
        )

        gate_probability = torch.sigmoid(
            gate_logits
        )

        residual_correction = (
            self.correction_head(
                shock_representation
            ).squeeze(-1)
        )

        final_forecast = (
            base_forecast
            + gate_probability
            * residual_correction
        )

        return {
            "gate_logits": gate_logits,
            "gate_probability": gate_probability,
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }

In [8]:
adapter = RetailShockAdapter(
    input_dim=int(
        adapter_config["input_dim"]
    ),
    encoder_width=int(
        adapter_config["encoder_width"]
    ),
    correction_hidden_width=int(
        adapter_config[
            "correction_hidden_width"
        ]
    ),
    extreme_prior=float(
        phase07_summary[
            "training_extreme_rate"
        ]
    ),
).to(DEVICE)

initial_state = torch.load(
    PHASE07_INITIAL_MODEL_DIR
    / "adapter_initial_state.pt",
    map_location=DEVICE,
    weights_only=True,
)

adapter.load_state_dict(
    initial_state
)

adapter_trainable_parameters = sum(
    parameter.numel()
    for parameter in adapter.parameters()
    if parameter.requires_grad
)

assert adapter_trainable_parameters == 2466

## Fixed loss-reference scales

In [9]:
training_base_mse = float(
    np.mean(
        np.square(
            np.asarray(
                train_residual_targets,
                dtype=np.float64,
            )
        )
    )
)

training_extreme_prior = float(
    phase07_summary[
        "training_extreme_rate"
    ]
)

gate_prior_entropy = float(
    -training_extreme_prior
    * math.log(
        training_extreme_prior
    )
    - (
        1.0
        - training_extreme_prior
    )
    * math.log(
        1.0
        - training_extreme_prior
    )
)

assert training_base_mse > 0.0
assert gate_prior_entropy > 0.0

print(
    "Training HP-TTM reference MSE:",
    training_base_mse,
)

print(
    "Gate-prior entropy:",
    gate_prior_entropy,
)

Training HP-TTM reference MSE: 5.181775591645961
Gate-prior entropy: 0.18974362087589505


## Validation baseline decomposition

In [10]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(
        np.float64,
        copy=False,
    )

    predicted_64 = predicted.astype(
        np.float64,
        copy=False,
    )

    error = predicted_64 - actual_64
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    mse = float(
        squared_error.mean()
    )

    rmse = float(
        np.sqrt(mse)
    )

    mae = float(
        absolute_error.mean()
    )

    denominator = float(
        np.abs(
            actual_64
        ).sum()
    )

    wape = float(
        absolute_error.sum()
        / denominator
    )

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

def masked_mse(
    actual: np.ndarray,
    predicted: np.ndarray,
    mask: np.ndarray,
) -> float:
    assert mask.any()

    errors = (
        predicted.astype(
            np.float64,
            copy=False,
        )
        - actual.astype(
            np.float64,
            copy=False,
        )
    )

    return float(
        np.square(
            errors[mask]
        ).mean()
    )

validation_extreme_mask = (
    np.asarray(
        validation_extreme_labels
    ) == 1
)

validation_normal_mask = (
    ~validation_extreme_mask
)

hp_validation_metrics = (
    forecasting_metrics(
        validation_targets,
        np.asarray(
            validation_base_forecast,
            dtype=np.float32,
        ),
    )
)

hp_validation_extreme_mse = (
    masked_mse(
        validation_targets,
        np.asarray(
            validation_base_forecast,
            dtype=np.float32,
        ),
        validation_extreme_mask,
    )
)

hp_validation_normal_mse = (
    masked_mse(
        validation_targets,
        np.asarray(
            validation_base_forecast,
            dtype=np.float32,
        ),
        validation_normal_mask,
    )
)

assert np.isclose(
    hp_validation_metrics["mse"],
    phase04_summary[
        "head_probed_metrics"
    ]["mse"],
    rtol=5e-3,
    atol=5e-3,
)

## Training configuration and checkpoint state

In [11]:
TRAINING_BATCH_SIZE = 512
INITIAL_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6
SCHEDULER_PATIENCE = 2
GRADIENT_CLIP_NORM = 1.0
CHECKPOINT_INTERVAL_STEPS = 50

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_training_state.pt"
)

BEST_BALANCED_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_balanced_state.pt"
)

BEST_OVERALL_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_overall_state.pt"
)

BEST_EXTREME_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_extreme_state.pt"
)

training_config = {
    "seed": BASE_SEED,
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": FEATURE_CONTRACT_SHA256,
    "crossfeed_contract_sha256": CROSSFEED_CONTRACT_SHA256,
    "extreme_contract_sha256": EXTREME_CONTRACT_SHA256,
    "architecture_contract_sha256": ARCHITECTURE_CONTRACT_SHA256,
    "batch_size": TRAINING_BATCH_SIZE,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "checkpoint_interval_steps": CHECKPOINT_INTERVAL_STEPS,
    "forecast_loss": "MSE / frozen_HP_training_MSE",
    "gate_loss": "BCEWithLogits / training_extreme_prior_entropy",
    "total_loss": "normalized_forecast_loss + normalized_gate_loss",
    "primary_checkpoint_selection": (
        "mean(EA_over_HP_overall_MSE_ratio, "
        "EA_over_HP_extreme_MSE_ratio)"
    ),
}

training_config_sha256 = hashlib.sha256(
    json.dumps(
        training_config,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

## Batch tensor construction

In [12]:
alignment_weight_tensor = torch.from_numpy(
    selected_weight
).to(
    DEVICE
)

def build_batch_tensors(
    indices: np.ndarray,
    *,
    training: bool,
) -> dict[str, torch.Tensor]:
    if training:
        decoder_source = train_decoder_hidden
        recent_source = train_recent_features
        future_source = train_future_covariates
        base_source = train_base_forecast
        target_source = training_targets
        extreme_source = train_extreme_labels
    else:
        decoder_source = validation_decoder_hidden
        recent_source = validation_recent_features
        future_source = validation_future_covariates
        base_source = validation_base_forecast
        target_source = validation_targets
        extreme_source = validation_extreme_labels

    raw_decoder = torch.from_numpy(
        np.asarray(
            decoder_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    recent = torch.from_numpy(
        np.asarray(
            recent_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    future = torch.from_numpy(
        np.asarray(
            future_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    base = torch.from_numpy(
        np.asarray(
            base_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    target = torch.from_numpy(
        np.asarray(
            target_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    extreme = torch.from_numpy(
        np.asarray(
            extreme_source[indices],
            dtype=np.float32,
        )
    ).to(DEVICE)

    decoder_features = torch.einsum(
        "bpk,hpk->bhk",
        raw_decoder[:, 0, :, :],
        alignment_weight_tensor,
    )

    return {
        "recent": recent,
        "decoder": decoder_features,
        "future": future,
        "base": base,
        "target": target,
        "extreme": extreme,
    }

## Validation evaluator

In [13]:
def evaluate_adapter(
    model: nn.Module,
) -> dict[str, Any]:
    model.eval()

    forecast_output = np.empty(
        (
            VALIDATION_SAMPLE_COUNT,
            FORECAST_HORIZON,
        ),
        dtype=np.float32,
    )

    gate_probability_output = np.empty(
        (
            VALIDATION_SAMPLE_COUNT,
            FORECAST_HORIZON,
        ),
        dtype=np.float32,
    )

    correction_output = np.empty(
        (
            VALIDATION_SAMPLE_COUNT,
            FORECAST_HORIZON,
        ),
        dtype=np.float32,
    )

    with torch.no_grad():
        for start in range(
            0,
            VALIDATION_SAMPLE_COUNT,
            TRAINING_BATCH_SIZE,
        ):
            end = min(
                start
                + TRAINING_BATCH_SIZE,
                VALIDATION_SAMPLE_COUNT,
            )

            indices = np.arange(
                start,
                end,
                dtype=np.int64,
            )

            batch = build_batch_tensors(
                indices,
                training=False,
            )

            output = model(
                recent_features=batch["recent"],
                decoder_features=batch["decoder"],
                future_covariates=batch["future"],
                base_forecast=batch["base"],
            )

            forecast_output[start:end] = (
                output["final_forecast"]
                .detach()
                .cpu()
                .numpy()
            )

            gate_probability_output[start:end] = (
                output["gate_probability"]
                .detach()
                .cpu()
                .numpy()
            )

            correction_output[start:end] = (
                output["residual_correction"]
                .detach()
                .cpu()
                .numpy()
            )

            del batch, output

    overall_metrics = forecasting_metrics(
        validation_targets,
        forecast_output,
    )

    extreme_mse = masked_mse(
        validation_targets,
        forecast_output,
        validation_extreme_mask,
    )

    normal_mse = masked_mse(
        validation_targets,
        forecast_output,
        validation_normal_mask,
    )

    overall_ratio = (
        overall_metrics["mse"]
        / hp_validation_metrics["mse"]
    )

    extreme_ratio = (
        extreme_mse
        / hp_validation_extreme_mse
    )

    balanced_score = float(
        0.5
        * (
            overall_ratio
            + extreme_ratio
        )
    )

    true_labels = (
        np.asarray(
            validation_extreme_labels,
            dtype=np.uint8,
        ).ravel()
    )

    gate_scores = (
        gate_probability_output.ravel()
    )

    gate_predictions = (
        gate_scores >= 0.5
    ).astype(
        np.uint8
    )

    gate_auroc = float(
        roc_auc_score(
            true_labels,
            gate_scores,
        )
    )

    gate_auprc = float(
        average_precision_score(
            true_labels,
            gate_scores,
        )
    )

    (
        gate_precision,
        gate_recall,
        gate_f1,
        _,
    ) = precision_recall_fscore_support(
        true_labels,
        gate_predictions,
        average="binary",
        zero_division=0,
    )

    return {
        "forecast": forecast_output,
        "gate_probability": gate_probability_output,
        "residual_correction": correction_output,
        "overall_metrics": overall_metrics,
        "extreme_mse": float(
            extreme_mse
        ),
        "normal_mse": float(
            normal_mse
        ),
        "overall_mse_ratio_vs_hp": float(
            overall_ratio
        ),
        "extreme_mse_ratio_vs_hp": float(
            extreme_ratio
        ),
        "balanced_score": balanced_score,
        "gate_auroc": gate_auroc,
        "gate_auprc": gate_auprc,
        "gate_precision_at_0_5": float(
            gate_precision
        ),
        "gate_recall_at_0_5": float(
            gate_recall
        ),
        "gate_f1_at_0_5": float(
            gate_f1
        ),
        "mean_gate_extreme": float(
            gate_probability_output[
                validation_extreme_mask
            ].mean(
                dtype=np.float64
            )
        ),
        "mean_gate_normal": float(
            gate_probability_output[
                validation_normal_mask
            ].mean(
                dtype=np.float64
            )
        ),
        "mean_abs_correction_extreme": float(
            np.abs(
                correction_output[
                    validation_extreme_mask
                ]
            ).mean(
                dtype=np.float64
            )
        ),
        "mean_abs_correction_normal": float(
            np.abs(
                correction_output[
                    validation_normal_mask
                ]
            ).mean(
                dtype=np.float64
            )
        ),
    }

## Initialize or resume optimization

In [14]:
optimizer = torch.optim.AdamW(
    adapter.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=SCHEDULER_PATIENCE,
    min_lr=1e-5,
)

forecast_loss_function = nn.MSELoss()
gate_loss_function = nn.BCEWithLogitsLoss()

start_epoch = 0
start_batch_index = 0
global_step = 0

best_balanced_score = float("inf")
best_overall_mse = float("inf")
best_extreme_mse = float("inf")

best_balanced_epoch = None
best_overall_epoch = None
best_extreme_epoch = None

early_stopping_counter = 0
training_history_records: list[
    dict[str, Any]
] = []

def capture_rng_state() -> dict[str, Any]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all(),
    }

def restore_rng_state(
    state: dict[str, Any],
) -> None:
    random.setstate(
        state["python"]
    )
    np.random.set_state(
        state["numpy"]
    )
    torch.set_rng_state(
        state["torch_cpu"]
    )
    torch.cuda.set_rng_state_all(
        state["torch_cuda"]
    )

def save_training_checkpoint(
    *,
    path: Path,
    epoch: int,
    next_batch_index: int,
) -> None:
    payload = {
        "model_state_dict": adapter.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": int(epoch),
        "next_batch_index": int(next_batch_index),
        "global_step": int(global_step),
        "best_balanced_score": float(best_balanced_score),
        "best_overall_mse": float(best_overall_mse),
        "best_extreme_mse": float(best_extreme_mse),
        "best_balanced_epoch": best_balanced_epoch,
        "best_overall_epoch": best_overall_epoch,
        "best_extreme_epoch": best_extreme_epoch,
        "early_stopping_counter": int(early_stopping_counter),
        "training_history_records": training_history_records,
        "training_config_sha256": training_config_sha256,
        "rng_state": capture_rng_state(),
    }

    temporary_path = path.with_suffix(
        ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        path
    )

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    assert (
        checkpoint[
            "training_config_sha256"
        ]
        == training_config_sha256
    )

    adapter.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    scheduler.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )

    start_epoch = int(
        checkpoint["epoch"]
    )

    start_batch_index = int(
        checkpoint[
            "next_batch_index"
        ]
    )

    global_step = int(
        checkpoint[
            "global_step"
        ]
    )

    best_balanced_score = float(
        checkpoint[
            "best_balanced_score"
        ]
    )

    best_overall_mse = float(
        checkpoint[
            "best_overall_mse"
        ]
    )

    best_extreme_mse = float(
        checkpoint[
            "best_extreme_mse"
        ]
    )

    best_balanced_epoch = checkpoint[
        "best_balanced_epoch"
    ]

    best_overall_epoch = checkpoint[
        "best_overall_epoch"
    ]

    best_extreme_epoch = checkpoint[
        "best_extreme_epoch"
    ]

    early_stopping_counter = int(
        checkpoint[
            "early_stopping_counter"
        ]
    )

    training_history_records = list(
        checkpoint[
            "training_history_records"
        ]
    )

    restore_rng_state(
        checkpoint[
            "rng_state"
        ]
    )

print(
    "Training start epoch:",
    start_epoch,
)

print(
    "Training start batch:",
    start_batch_index,
)

print(
    "Global step:",
    global_step,
)

Training start epoch: 0
Training start batch: 0
Global step: 0


## EA-TTM training loop

In [15]:
training_start_time = time.perf_counter()

num_training_batches = math.ceil(
    TRAINING_SAMPLE_COUNT
    / TRAINING_BATCH_SIZE
)

for epoch in range(
    start_epoch,
    MAX_EPOCHS,
):
    adapter.train()

    epoch_rng = np.random.default_rng(
        BASE_SEED + epoch
    )

    permutation = epoch_rng.permutation(
        TRAINING_SAMPLE_COUNT
    )

    first_batch = (
        start_batch_index
        if epoch == start_epoch
        else 0
    )

    epoch_total_loss_sum = 0.0
    epoch_forecast_loss_sum = 0.0
    epoch_gate_loss_sum = 0.0
    epoch_sample_count = 0

    for batch_index in range(
        first_batch,
        num_training_batches,
    ):
        batch_start = (
            batch_index
            * TRAINING_BATCH_SIZE
        )

        batch_end = min(
            batch_start
            + TRAINING_BATCH_SIZE,
            TRAINING_SAMPLE_COUNT,
        )

        batch_indices = permutation[
            batch_start:batch_end
        ]

        batch = build_batch_tensors(
            batch_indices,
            training=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        output = adapter(
            recent_features=batch["recent"],
            decoder_features=batch["decoder"],
            future_covariates=batch["future"],
            base_forecast=batch["base"],
        )

        raw_forecast_loss = (
            forecast_loss_function(
                output["final_forecast"],
                batch["target"],
            )
        )

        raw_gate_loss = (
            gate_loss_function(
                output["gate_logits"],
                batch["extreme"],
            )
        )

        normalized_forecast_loss = (
            raw_forecast_loss
            / training_base_mse
        )

        normalized_gate_loss = (
            raw_gate_loss
            / gate_prior_entropy
        )

        total_loss = (
            normalized_forecast_loss
            + normalized_gate_loss
        )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            adapter.parameters(),
            GRADIENT_CLIP_NORM,
        )

        optimizer.step()

        current_batch_size = (
            batch_end
            - batch_start
        )

        epoch_total_loss_sum += (
            float(
                total_loss.detach().item()
            )
            * current_batch_size
        )

        epoch_forecast_loss_sum += (
            float(
                raw_forecast_loss.detach().item()
            )
            * current_batch_size
        )

        epoch_gate_loss_sum += (
            float(
                raw_gate_loss.detach().item()
            )
            * current_batch_size
        )

        epoch_sample_count += (
            current_batch_size
        )

        global_step += 1

        if (
            global_step
            % CHECKPOINT_INTERVAL_STEPS
            == 0
        ):
            save_training_checkpoint(
                path=LAST_CHECKPOINT_PATH,
                epoch=epoch,
                next_batch_index=(
                    batch_index + 1
                ),
            )

        del (
            batch,
            output,
            raw_forecast_loss,
            raw_gate_loss,
            normalized_forecast_loss,
            normalized_gate_loss,
            total_loss,
        )

    mean_total_loss = (
        epoch_total_loss_sum
        / epoch_sample_count
    )

    mean_forecast_mse = (
        epoch_forecast_loss_sum
        / epoch_sample_count
    )

    mean_gate_bce = (
        epoch_gate_loss_sum
        / epoch_sample_count
    )

    validation_result = (
        evaluate_adapter(
            adapter
        )
    )

    current_balanced_score = (
        validation_result[
            "balanced_score"
        ]
    )

    current_overall_mse = (
        validation_result[
            "overall_metrics"
        ]["mse"]
    )

    current_extreme_mse = (
        validation_result[
            "extreme_mse"
        ]
    )

    current_learning_rate = float(
        optimizer.param_groups[0][
            "lr"
        ]
    )

    training_history_records.append(
        {
            "epoch": epoch + 1,
            "train_total_loss": mean_total_loss,
            "train_forecast_mse": mean_forecast_mse,
            "train_gate_bce": mean_gate_bce,
            "validation_mse": current_overall_mse,
            "validation_extreme_mse": current_extreme_mse,
            "validation_normal_mse": validation_result["normal_mse"],
            "validation_balanced_score": current_balanced_score,
            "gate_auroc": validation_result["gate_auroc"],
            "gate_auprc": validation_result["gate_auprc"],
            "gate_f1_at_0_5": validation_result["gate_f1_at_0_5"],
            "learning_rate": current_learning_rate,
            "global_step": global_step,
        }
    )

    balanced_improved = (
        current_balanced_score
        < best_balanced_score
        - 1e-8
    )

    if balanced_improved:
        best_balanced_score = float(
            current_balanced_score
        )

        best_balanced_epoch = (
            epoch + 1
        )

        early_stopping_counter = 0

        save_training_checkpoint(
            path=BEST_BALANCED_CHECKPOINT_PATH,
            epoch=epoch + 1,
            next_batch_index=0,
        )
    else:
        early_stopping_counter += 1

    if (
        current_overall_mse
        < best_overall_mse
        - 1e-8
    ):
        best_overall_mse = float(
            current_overall_mse
        )

        best_overall_epoch = (
            epoch + 1
        )

        save_training_checkpoint(
            path=BEST_OVERALL_CHECKPOINT_PATH,
            epoch=epoch + 1,
            next_batch_index=0,
        )

    if (
        current_extreme_mse
        < best_extreme_mse
        - 1e-8
    ):
        best_extreme_mse = float(
            current_extreme_mse
        )

        best_extreme_epoch = (
            epoch + 1
        )

        save_training_checkpoint(
            path=BEST_EXTREME_CHECKPOINT_PATH,
            epoch=epoch + 1,
            next_batch_index=0,
        )

    scheduler.step(
        current_balanced_score
    )

    save_training_checkpoint(
        path=LAST_CHECKPOINT_PATH,
        epoch=epoch + 1,
        next_batch_index=0,
    )

    start_batch_index = 0

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train MSE={mean_forecast_mse:.6f} | "
        f"val MSE={current_overall_mse:.6f} | "
        f"extreme MSE={current_extreme_mse:.6f} | "
        f"balanced={current_balanced_score:.6f} | "
        f"AUPRC={validation_result['gate_auprc']:.4f}"
    )

    if (
        early_stopping_counter
        >= EARLY_STOPPING_PATIENCE
    ):
        break

training_runtime_seconds = (
    time.perf_counter()
    - training_start_time
)

assert (
    BEST_BALANCED_CHECKPOINT_PATH.exists()
)

Epoch 01 | train MSE=5.181080 | val MSE=4.377969 | extreme MSE=39.714741 | balanced=0.996120 | AUPRC=0.1752
Epoch 02 | train MSE=5.179660 | val MSE=4.376940 | extreme MSE=39.648612 | balanced=0.995176 | AUPRC=0.1854
Epoch 03 | train MSE=5.178667 | val MSE=4.376192 | extreme MSE=39.619350 | balanced=0.994725 | AUPRC=0.1857
Epoch 04 | train MSE=5.177543 | val MSE=4.375692 | extreme MSE=39.550348 | balanced=0.993805 | AUPRC=0.1886
Epoch 05 | train MSE=5.175566 | val MSE=4.373959 | extreme MSE=39.640880 | balanced=0.994739 | AUPRC=0.1919
Epoch 06 | train MSE=5.174366 | val MSE=4.372720 | extreme MSE=39.260752 | balanced=0.989846 | AUPRC=0.1919
Epoch 07 | train MSE=5.172520 | val MSE=4.370899 | extreme MSE=39.437539 | balanced=0.991848 | AUPRC=0.1945
Epoch 08 | train MSE=5.171612 | val MSE=4.369002 | extreme MSE=39.366717 | balanced=0.990746 | AUPRC=0.1987
Epoch 09 | train MSE=5.170920 | val MSE=4.368909 | extreme MSE=39.294134 | balanced=0.989828 | AUPRC=0.2021
Epoch 10 | train MSE=5.16903

## Restore primary balanced checkpoint

In [16]:
primary_checkpoint = torch.load(
    BEST_BALANCED_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

adapter.load_state_dict(
    primary_checkpoint[
        "model_state_dict"
    ]
)

adapter.eval()

primary_validation_result = (
    evaluate_adapter(
        adapter
    )
)

primary_epoch = int(
    primary_checkpoint["epoch"]
)

assert primary_epoch == best_balanced_epoch

## Final validation comparison

In [17]:
def improvement_percent(
    baseline: float,
    candidate: float,
) -> float:
    return float(
        100.0
        * (
            baseline
            - candidate
        )
        / baseline
    )

ea_validation_metrics = (
    primary_validation_result[
        "overall_metrics"
    ]
)

ea_extreme_mse = float(
    primary_validation_result[
        "extreme_mse"
    ]
)

ea_normal_mse = float(
    primary_validation_result[
        "normal_mse"
    ]
)

validation_comparison = {
    "overall_mse_improvement_percent": (
        improvement_percent(
            hp_validation_metrics["mse"],
            ea_validation_metrics["mse"],
        )
    ),
    "overall_rmse_improvement_percent": (
        improvement_percent(
            hp_validation_metrics["rmse"],
            ea_validation_metrics["rmse"],
        )
    ),
    "overall_mae_improvement_percent": (
        improvement_percent(
            hp_validation_metrics["mae"],
            ea_validation_metrics["mae"],
        )
    ),
    "overall_wape_improvement_percent": (
        improvement_percent(
            hp_validation_metrics["wape"],
            ea_validation_metrics["wape"],
        )
    ),
    "extreme_mse_improvement_percent": (
        improvement_percent(
            hp_validation_extreme_mse,
            ea_extreme_mse,
        )
    ),
    "normal_mse_improvement_percent": (
        improvement_percent(
            hp_validation_normal_mse,
            ea_normal_mse,
        )
    ),
}

In [18]:
comparison_table = pd.DataFrame(
    [
        {
            "model": "Frozen HP-TTM",
            **hp_validation_metrics,
            "extreme_mse": hp_validation_extreme_mse,
            "normal_mse": hp_validation_normal_mse,
        },
        {
            "model": "EA-TTM",
            **ea_validation_metrics,
            "extreme_mse": ea_extreme_mse,
            "normal_mse": ea_normal_mse,
        },
    ]
)

comparison_table

,model,mse,rmse,mae,wape,extreme_mse,normal_mse
0,Frozen HP-TTM,4.381208,2.093133,1.007348,0.726575,39.995519,2.216475
1,EA-TTM,4.368124,2.090006,1.022395,0.737428,39.210961,2.250284


## Save trained adapter and validation evidence

In [19]:
FINAL_ADAPTER_STATE_PATH = (
    FINAL_ADAPTER_DIR
    / "adapter_state.pt"
)

FINAL_ADAPTER_CONFIG_PATH = (
    FINAL_ADAPTER_DIR
    / "adapter_config.json"
)

TRAINING_HISTORY_PATH = (
    RESULTS_TABLE_DIR
    / "08_EA_TTM_TRAINING_HISTORY.csv"
)

VALIDATION_COMPARISON_PATH = (
    RESULTS_TABLE_DIR
    / "08_EA_TTM_VALIDATION_COMPARISON.csv"
)

VALIDATION_OUTPUTS_PATH = (
    RUN_DIR
    / "08_VALIDATION_OUTPUTS.npz"
)

TRAINING_CONTRACT_PATH = (
    METADATA_DIR
    / "08_TRAINING_CONTRACT.json"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "08_RUN_SUMMARY.json"
)

torch.save(
    adapter.state_dict(),
    FINAL_ADAPTER_STATE_PATH,
)

final_adapter_config = {
    **adapter_config,
    "source_initial_architecture_contract_sha256": (
        ARCHITECTURE_CONTRACT_SHA256
    ),
    "training_config_sha256": (
        training_config_sha256
    ),
    "selected_checkpoint": "best_balanced",
    "selected_epoch": primary_epoch,
}

FINAL_ADAPTER_CONFIG_PATH.write_text(
    json.dumps(
        final_adapter_config,
        indent=2,
    ),
    encoding="utf-8",
)

pd.DataFrame(
    training_history_records
).to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
)

comparison_table.to_csv(
    VALIDATION_COMPARISON_PATH,
    index=False,
)

np.savez_compressed(
    VALIDATION_OUTPUTS_PATH,
    actual=validation_targets.astype(
        np.float32
    ),
    hp_ttm_forecast=np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    ),
    ea_ttm_forecast=primary_validation_result[
        "forecast"
    ].astype(
        np.float32
    ),
    gate_probability=primary_validation_result[
        "gate_probability"
    ].astype(
        np.float32
    ),
    residual_correction=primary_validation_result[
        "residual_correction"
    ].astype(
        np.float32
    ),
    extreme_label=np.asarray(
        validation_extreme_labels,
        dtype=np.uint8,
    ),
)

TRAINING_CONTRACT_PATH.write_text(
    json.dumps(
        {
            **training_config,
            "training_config_sha256": training_config_sha256,
            "training_hp_reference_mse": training_base_mse,
            "gate_prior_entropy": gate_prior_entropy,
            "primary_epoch": primary_epoch,
        },
        indent=2,
    ),
    encoding="utf-8",
)

1310

In [20]:
run_summary = {
    "status": "PHASE_08_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": FEATURE_CONTRACT_SHA256,
    "crossfeed_contract_sha256": CROSSFEED_CONTRACT_SHA256,
    "extreme_contract_sha256": EXTREME_CONTRACT_SHA256,
    "architecture_contract_sha256": ARCHITECTURE_CONTRACT_SHA256,
    "training_config_sha256": training_config_sha256,
    "adapter_trainable_parameters": adapter_trainable_parameters,
    "training_batch_size": TRAINING_BATCH_SIZE,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "best_balanced_epoch": best_balanced_epoch,
    "best_overall_epoch": best_overall_epoch,
    "best_extreme_epoch": best_extreme_epoch,
    "primary_selected_epoch": primary_epoch,
    "global_step": int(global_step),
    "training_runtime_seconds": float(
        training_runtime_seconds
    ),
    "hp_ttm_validation_metrics": hp_validation_metrics,
    "hp_ttm_validation_extreme_mse": hp_validation_extreme_mse,
    "hp_ttm_validation_normal_mse": hp_validation_normal_mse,
    "ea_ttm_validation_metrics": ea_validation_metrics,
    "ea_ttm_validation_extreme_mse": ea_extreme_mse,
    "ea_ttm_validation_normal_mse": ea_normal_mse,
    "ea_ttm_balanced_score": float(
        primary_validation_result[
            "balanced_score"
        ]
    ),
    "validation_improvement_vs_hp": validation_comparison,
    "gate_metrics": {
        "auroc": float(
            primary_validation_result[
                "gate_auroc"
            ]
        ),
        "auprc": float(
            primary_validation_result[
                "gate_auprc"
            ]
        ),
        "precision_at_0_5": float(
            primary_validation_result[
                "gate_precision_at_0_5"
            ]
        ),
        "recall_at_0_5": float(
            primary_validation_result[
                "gate_recall_at_0_5"
            ]
        ),
        "f1_at_0_5": float(
            primary_validation_result[
                "gate_f1_at_0_5"
            ]
        ),
        "mean_gate_extreme": float(
            primary_validation_result[
                "mean_gate_extreme"
            ]
        ),
        "mean_gate_normal": float(
            primary_validation_result[
                "mean_gate_normal"
            ]
        ),
    },
    "correction_diagnostics": {
        "mean_abs_correction_extreme": float(
            primary_validation_result[
                "mean_abs_correction_extreme"
            ]
        ),
        "mean_abs_correction_normal": float(
            primary_validation_result[
                "mean_abs_correction_normal"
            ]
        ),
    },
    "locked_final_test_used": False,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

2458

In [21]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 08 — EA-TTM adapter training completed.",
    "",
    "## Primary checkpoint",
    "- Selection: balanced overall/extreme validation score",
    f"- Epoch: {primary_epoch}",
    f"- Trainable adapter parameters: {adapter_trainable_parameters:,}",
    "",
    "## Validation",
    f"- HP-TTM MSE: {hp_validation_metrics['mse']:.6f}",
    f"- EA-TTM MSE: {ea_validation_metrics['mse']:.6f}",
    f"- HP-TTM extreme MSE: {hp_validation_extreme_mse:.6f}",
    f"- EA-TTM extreme MSE: {ea_extreme_mse:.6f}",
    f"- Gate AUPRC: {primary_validation_result['gate_auprc']:.6f}",
    "",
    "## Test status",
    "- Locked final-test interval remains unused",
    "",
    "## Next phase",
    (
        "Run ablation and mechanism diagnostics on validation, "
        "then freeze all model-development decisions before the "
        "single locked final-test evaluation."
    ),
]

status_path.write_text(
    "\n".join(
        status_lines
    ),
    encoding="utf-8",
)

584

## Phase validation

Phase 08 passes only if:

$$
\begin{aligned}
&\text{only the 2,466-parameter adapter is trainable}
\\
&\land\
\text{a primary balanced-validation checkpoint exists}
\\
&\land\
\text{all EA-TTM validation outputs are finite}
\\
&\land\
0\le g_h\le1
\\
&\land\
\text{trained adapter and checkpoints are persisted}
\\
&\land\
\text{locked final-test demand remains unused}.
\end{aligned}
$$

In [22]:
phase_validation = {
    "adapter_parameter_count_valid": (
        adapter_trainable_parameters
        == 2466
    ),
    "primary_checkpoint_exists": (
        BEST_BALANCED_CHECKPOINT_PATH.exists()
    ),
    "trained_adapter_saved": (
        FINAL_ADAPTER_STATE_PATH.exists()
    ),
    "forecast_finite": bool(
        np.isfinite(
            primary_validation_result[
                "forecast"
            ]
        ).all()
    ),
    "gate_finite": bool(
        np.isfinite(
            primary_validation_result[
                "gate_probability"
            ]
        ).all()
    ),
    "gate_range_valid": bool(
        (
            primary_validation_result[
                "gate_probability"
            ] >= 0.0
        ).all()
        and (
            primary_validation_result[
                "gate_probability"
            ] <= 1.0
        ).all()
    ),
    "validation_outputs_saved": (
        VALIDATION_OUTPUTS_PATH.exists()
    ),
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
    "locked_final_test_unused": True,
}

assert all(
    phase_validation.values()
), (
    f"Phase 08 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'adapter_parameter_count_valid': True,
 'primary_checkpoint_exists': True,
 'trained_adapter_saved': True,
 'forecast_finite': True,
 'gate_finite': True,
 'gate_range_valid': True,
 'validation_outputs_saved': True,
 'run_summary_saved': True,
 'locked_final_test_unused': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [23]:
return_block = {
    "status": "PHASE_08_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "training_config_sha256": training_config_sha256,
    "adapter_trainable_parameters": adapter_trainable_parameters,
    "training_batch_size": TRAINING_BATCH_SIZE,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "best_balanced_epoch": best_balanced_epoch,
    "best_overall_epoch": best_overall_epoch,
    "best_extreme_epoch": best_extreme_epoch,
    "primary_selected_epoch": primary_epoch,
    "global_step": int(global_step),
    "training_runtime_seconds": float(
        training_runtime_seconds
    ),
    "hp_ttm_validation": {
        "overall_metrics": hp_validation_metrics,
        "extreme_mse": hp_validation_extreme_mse,
        "normal_mse": hp_validation_normal_mse,
    },
    "ea_ttm_validation": {
        "overall_metrics": ea_validation_metrics,
        "extreme_mse": ea_extreme_mse,
        "normal_mse": ea_normal_mse,
        "balanced_score": float(
            primary_validation_result[
                "balanced_score"
            ]
        ),
    },
    "improvement_vs_hp_percent": (
        validation_comparison
    ),
    "gate_metrics": {
        "auroc": float(
            primary_validation_result[
                "gate_auroc"
            ]
        ),
        "auprc": float(
            primary_validation_result[
                "gate_auprc"
            ]
        ),
        "precision_at_0_5": float(
            primary_validation_result[
                "gate_precision_at_0_5"
            ]
        ),
        "recall_at_0_5": float(
            primary_validation_result[
                "gate_recall_at_0_5"
            ]
        ),
        "f1_at_0_5": float(
            primary_validation_result[
                "gate_f1_at_0_5"
            ]
        ),
        "mean_gate_extreme": float(
            primary_validation_result[
                "mean_gate_extreme"
            ]
        ),
        "mean_gate_normal": float(
            primary_validation_result[
                "mean_gate_normal"
            ]
        ),
    },
    "correction_diagnostics": {
        "mean_abs_correction_extreme": float(
            primary_validation_result[
                "mean_abs_correction_extreme"
            ]
        ),
        "mean_abs_correction_normal": float(
            primary_validation_result[
                "mean_abs_correction_normal"
            ]
        ),
    },
    "locked_final_test_used": False,
    "artifacts": {
        "trained_adapter_state": str(
            FINAL_ADAPTER_STATE_PATH
        ),
        "trained_adapter_config": str(
            FINAL_ADAPTER_CONFIG_PATH
        ),
        "last_checkpoint": str(
            LAST_CHECKPOINT_PATH
        ),
        "best_balanced_checkpoint": str(
            BEST_BALANCED_CHECKPOINT_PATH
        ),
        "best_overall_checkpoint": str(
            BEST_OVERALL_CHECKPOINT_PATH
        ),
        "best_extreme_checkpoint": str(
            BEST_EXTREME_CHECKPOINT_PATH
        ),
        "training_history": str(
            TRAINING_HISTORY_PATH
        ),
        "validation_comparison": str(
            VALIDATION_COMPARISON_PATH
        ),
        "validation_outputs": str(
            VALIDATION_OUTPUTS_PATH
        ),
        "training_contract": str(
            TRAINING_CONTRACT_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 08 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 08 RETURN BLOCK
{
  "status": "PHASE_08_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "training_config_sha256": "f6660977fc785413288b282bb6761e7ef82a0c90936aaf91d4aa0cf7a6383d6f",
  "adapter_trainable_parameters": 2466,
  "training_batch_size": 512,
  "initial_learning_rate": 0.001,
  "best_balanced_epoch": 15,
  "best_overall_epoch": 20,
  "best_extreme_epoch": 15,
  "primary_selected_epoch": 15,
  "global_step": 5019,
  "training_runtime_seconds": 116.82851426299999,
  "hp_ttm_validation": {
    "overall_metrics": {
      "mse": 4.381207550987759,
      "rmse": 2.0931334288543955,
      "mae": 1.00734836222702,
      "wape": 0.726575323464449
    },
    "extreme_mse": 39.99551897843197,
    "normal_mse": 2.2164752486230572
  },
  "ea_ttm_validation": {
    "overall_metrics": {
      "mse": 4.368124020733498,
      "rmse": 2.090005746579061